# 02 — Named Entity Recognition & Structured Data Extraction

**Goal:** load the `.txt` files saved by notebook 01, run spaCy NER, extract financial figures
(regex) and growth-driver sentences (keyword matching), and build one structured row per company
into `data/processed/company_extractions.csv`.

This notebook does NOT depend on notebook 01's variables still being in memory — it reads fresh
from disk, so you can run this any time after 01 has been run once.

In [1]:
import os

PROJECT_ROOT = r"C:\Users\Devanshi\financial-nlp-analyzer"
os.chdir(PROJECT_ROOT)
print("Working directory set to:", os.getcwd())

Working directory set to: C:\Users\Devanshi\financial-nlp-analyzer


## Setup

In [2]:
import os
import re
import pandas as pd
import spacy

nlp = spacy.load("en_core_web_sm")

EXTRACTED_DIR = "data/extracted_text"
PROCESSED_DIR = "data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

COMPANIES = ["jpmorgan", "caterpillar", "accenture", "pg", "infosys"]


## Load extracted text from disk

In [3]:
def load_section(company, section):
    """Loads a saved section .txt file. Returns None if it doesn't exist."""
    path = os.path.join(EXTRACTED_DIR, f"{company}_{section}.txt")
    if not os.path.exists(path):
        print(f"WARNING: {path} not found — skipping")
        return None
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

texts = {}
for company in COMPANIES:
    texts[company] = {
        "mdna": load_section(company, "mdna"),
        "risk": load_section(company, "risk"),
    }
    mdna_len = len(texts[company]["mdna"]) if texts[company]["mdna"] else 0
    risk_len = len(texts[company]["risk"]) if texts[company]["risk"] else 0
    print(f"{company}: mdna={mdna_len:,} chars, risk={risk_len:,} chars")


jpmorgan: mdna=662 chars, risk=136,029 chars
caterpillar: mdna=209,867 chars, risk=75,113 chars
accenture: mdna=46,256 chars, risk=91,511 chars
pg: mdna=91,326 chars, risk=38,613 chars
infosys: mdna=5,092 chars, risk=25,453 chars


## Extraction functions (Phase 6)

In [4]:
def extract_entities(text):
    """Runs spaCy NER, returns list of (entity_text, entity_label) tuples."""
    doc = nlp(text)
    return [(ent.text, ent.label_) for ent in doc.ents]


def extract_financial_figures(text):
    """Regex-based extraction of money amounts and percentages."""
    money_pattern = r'\$[\d,]+(?:\.\d+)?\s?(?:million|billion|thousand)?'
    percent_pattern = r'\d+(?:\.\d+)?%'
    rupee_pattern = r'₹\s?[\d,]+(?:\.\d+)?\s?(?:crore|lakh|million|billion)?'

    return {
        "money_figures": re.findall(money_pattern, text),
        "percentages": re.findall(percent_pattern, text),
        "rupee_figures": re.findall(rupee_pattern, text),
    }


GROWTH_KEYWORDS = ["driven by", "due to", "primarily attributable to", "as a result of",
                    "increased", "decreased", "grew", "declined"]

def extract_driver_sentences(text):
    """Pulls out sentences that likely explain a growth/decline driver."""
    doc = nlp(text)
    return [sent.text.strip() for sent in doc.sents
            if any(kw in sent.text.lower() for kw in GROWTH_KEYWORDS)]


## spaCy has a max text length — long MD&A sections may need truncation

spaCy's default `nlp.max_length` is ~1,000,000 characters, which is usually enough — but if you hit
a `ValueError` about text length, this cell raises the limit safely.

In [5]:
nlp.max_length = 2_000_000  # raise if needed for very long documents


## Build one structured record per company

In [6]:
def build_company_record(company_name, mdna_text, risk_text):
    if mdna_text is None:
        print(f"Skipping {company_name} — MD&A text missing (check notebook 01 output)")
        return None

    entities = extract_entities(mdna_text)
    figures = extract_financial_figures(mdna_text)
    drivers = extract_driver_sentences(mdna_text)

    return {
        "company": company_name,
        "mdna_length": len(mdna_text),
        "risk_length": len(risk_text) if risk_text else 0,
        "num_orgs_mentioned": len([e for e in entities if e[1] == "ORG"]),
        "num_money_figures": len(figures["money_figures"]),
        "num_percentages": len(figures["percentages"]),
        "num_rupee_figures": len(figures["rupee_figures"]),
        "num_growth_driver_sentences": len(drivers),
        "growth_driver_sentences": " | ".join(drivers[:10]),  # cap for CSV readability
        "sample_money_figures": ", ".join(figures["money_figures"][:10]),
    }


records = []
for company in COMPANIES:
    record = build_company_record(company, texts[company]["mdna"], texts[company]["risk"])
    if record:
        records.append(record)

df = pd.DataFrame(records)
df


,company,mdna_length,risk_length,num_orgs_mentioned,num_money_figures,num_percentages,num_rupee_figures,num_growth_driver_sentences,growth_driver_sentences,sample_money_figures
0,jpmorgan,662,136029,3,0,0,0,0,,
1,caterpillar,209867,75113,711,166,47,0,102,Emissions compliance in developing markets is ...,"$30.0 billion, $27.5 billion, $8.0 billion, $1..."
2,accenture,46256,91511,97,42,46,0,41,For additional information regarding business ...,"$438 million, $1,063 million, $253 million, $6..."
3,pg,91326,38613,251,103,116,0,140,Risks and uncertainties to which our forward-l...,"$1.0 , $1.5 billion, $216 million, $750 millio..."
4,infosys,5092,25453,17,0,1,0,0,,


## Save results

In [7]:
out_path = os.path.join(PROCESSED_DIR, "company_extractions.csv")
df.to_csv(out_path, index=False)
print(f"Saved {out_path}")


Saved data/processed\company_extractions.csv


In [8]:
import os
print("Current working directory:", os.getcwd())
print("Absolute path of saved file:", os.path.abspath(out_path))
print("Does it exist there?", os.path.exists(out_path))

Current working directory: C:\Users\Devanshi\financial-nlp-analyzer
Absolute path of saved file: C:\Users\Devanshi\financial-nlp-analyzer\data\processed\company_extractions.csv
Does it exist there? True


In [9]:
import shutil

wrong_path = r"C:\Users\Devanshi\financial-nlp-analyzer\notebooks\data"
correct_path = r"C:\Users\Devanshi\financial-nlp-analyzer\data"

if os.path.exists(wrong_path):
    for root, dirs, files in os.walk(wrong_path):
        for file in files:
            src = os.path.join(root, file)
            rel_path = os.path.relpath(src, wrong_path)
            dst = os.path.join(correct_path, rel_path)
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.move(src, dst)
            print(f"Moved: {rel_path}")
    shutil.rmtree(wrong_path)
    print("\nCleaned up the stray notebooks/data folder.")
else:
    print("No stray folder found — nothing to move.")

No stray folder found — nothing to move.
